In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('csv').option('header',True).option('inferSchema',True).load("/FileStore/tables/ipl_over_data.csv")
df.display()

year,series_type,series_name,match_no,match_type,match_id,match_venue,match_status,match_winning_team,match_tie_breaker,match_toss,umpires,match_referee,third_umpires,match_datetime,team1_name,team2_name,team1_score,team1_wickets,team2_score,team2_wickets,team1_captain,team1_players,team1_bench,team1_support_staff,team2_captain,team2_players,team2_bench,team2_support_staff,over_no,over_total_runs,over_summary,over_batsman1_name,over_batsman1_curr_scr,over_batsman1_played_balls,over_batsman2_name,over_batsman2_curr_scr,over_batsman2_played_balls,over_bowler_name,over_bowler_bowled_overs,over_bowler_bowled_maidens,over_bowler_bowled_runs,over_bowler_bowled_wickets
2021,T20 League,Indian Premier League 2021,1st Match,T20 IPL,35612,"{'stadium': 'MA Chidambaram Stadium', 'city': 'Chennai', 'capacity': '50000', 'host_teams': 'Tamil Nadu, Chennai Super Kings'}",completed,Royal Challengers Bangalore,null,Royal Challengers Bangalore have won the toss and have opted to field,"KN Anantha Padmanabhan, Nitin Menon",Vengalil Narayanan Kutty,Chettithody Shamshuddin,2021-04-09 19:30:00+05:30,MI,RCB,159,9,160,8,null,[],"Nathan Coulter-Nile, Piyush Chawla, Dhawal Kulkarni, Saurabh Tiwary, Aditya Tare, Adam Milne, Jayant Yadav, Anmolpreet Singh, Quinton de Kock, Anukul Roy, Mohsin Khan, Arjun Tendulkar, Yudhvir Singh Charak, James Neesham",null,null,[],"Adam Zampa, Devdutt Padikkal, Sachin Baby, Navdeep Saini, Kane Richardson, Srikar Bharat, Pavan Deshpande, Finn Allen, Suyash Prabhudessai, Mohammed Azharuddeen",null,1,5,2 0 0 2 0 1,Rohit Sharma,5,6,Chris Lynn,0,0,Mohammed Siraj,1,0,5,0
2021,T20 League,Indian Premier League 2021,1st Match,T20 IPL,35612,"{'stadium': 'MA Chidambaram Stadium', 'city': 'Chennai', 'capacity': '50000', 'host_teams': 'Tamil Nadu, Chennai Super Kings'}",completed,Royal Challengers Bangalore,null,Royal Challengers Bangalore have won the toss and have opted to field,"KN Anantha Padmanabhan, Nitin Menon",Vengalil Narayanan Kutty,Chettithody Shamshuddin,2021-04-09 19:30:00+05:30,MI,RCB,159,9,160,8,null,[],"Nathan Coulter-Nile, Piyush Chawla, Dhawal Kulkarni, Saurabh Tiwary, Aditya Tare, Adam Milne, Jayant Yadav, Anmolpreet Singh, Quinton de Kock, Anukul Roy, Mohsin Khan, Arjun Tendulkar, Yudhvir Singh Charak, James Neesham",null,null,[],"Adam Zampa, Devdutt Padikkal, Sachin Baby, Navdeep Saini, Kane Richardson, Srikar Bharat, Pavan Deshpande, Finn Allen, Suyash Prabhudessai, Mohammed Azharuddeen",null,2,1,1 0 0 0 0 0,Chris Lynn,0,5,Rohit Sharma,6,7,Kyle Jamieson,1,0,1,0
2021,T20 League,Indian Premier League 2021,1st Match,T20 IPL,35612,"{'stadium': 'MA Chidambaram Stadium', 'city': 'Chennai', 'capacity': '50000', 'host_teams': 'Tamil Nadu, Chennai Super Kings'}",completed,Royal Challengers Bangalore,null,Royal Challengers Bangalore have won the toss and have opted to field,"KN Anantha Padmanabhan, Nitin Menon",Vengalil Narayanan Kutty,Chettithody Shamshuddin,2021-04-09 19:30:00+05:30,MI,RCB,159,9,160,8,null,[],"Nathan Coulter-Nile, Piyush Chawla, Dhawal Kulkarni, Saurabh Tiwary, Aditya Tare, Adam Milne, Jayant Yadav, Anmolpreet Singh, Quinton de Kock, Anukul Roy, Mohsin Khan, Arjun Tendulkar, Yudhvir Singh Charak, James Neesham",null,null,[],"Adam Zampa, Devdutt Padikkal, Sachin Baby, Navdeep Saini, Kane Richardson, Srikar Bharat, Pavan Deshpande, Finn Allen, Suyash Prabhudessai, Mohammed Azharuddeen",null,3,6,0 0 0 0 4 2,Rohit Sharma,12,13,Chris Lynn,0,5,Mohammed Siraj,2,0,11,0
2021,T20 League,Indian Premier League 2021,1st Match,T20 IPL,35612,"{'stadium': 'MA Chidambaram Stadium', 'city': 'Chennai', 'capacity': '50000', 'host_teams': 'Tamil Nadu, Chennai Super Kings'}",completed,Royal Challengers Bangalore,null,Royal Challengers Bangalore have won the toss and have opted to field,"KN Anantha Padmanabhan, Nitin Menon",Vengalil Narayanan Kutty,Chettithody Shamshuddin,2021-04-09 19:30:00+05:30,MI,RCB,159,9,160,8,null,[],"Nathan Coulter-Nile, Piyush Chawla, Dhawal Kulkarni, Saurabh Tiwary, Aditya Tare, Adam Milne, Jayant Yada

In [0]:
from pyspark.sql.functions import col, sum as _sum, max as _max
team_scores = df.select(
    col("year"),
    col("match_id"),
    col("team1_name").alias("team_name"),
    col("team1_score").cast("int").alias("team_score")
).union(
    df.select(
        col("year"),
        col("match_id"),
        col("team2_name").alias("team_name"),
        col("team2_score").cast("int").alias("team_score")
    )
).dropna(subset=["team_score", "team_name"])

avg_score_pivot = team_scores.groupBy("team_name").pivot("year").avg("team_score").orderBy("team_name")

max_score_pivot = team_scores.groupBy("team_name").pivot("year").agg(_max("team_score")).orderBy("team_name")

avg_score_pivot.display()
max_score_pivot.display()

team_name,2017,2018,2019,2020,2021,2022,2023,2024,2025
CSK,null,175.62540192926045,152.8356973995272,157.1425925925926,171.66829268292682,164.71641791044777,181.44444444444446,180.64154411764707,151.9240506329114
DC,161.7017208413002,170.59574468085106,157.96596434359805,163.85585585585585,155.4126213592233,168.3599257884972,155.6304347826087,186.8236397748593,211.0
GL,172.3548387096774,null,null,null,null,null,null,null,null
GT,null,null,null,null,null,166.95153473344104,180.20948012232415,173.02678571428572,232.0
KKR,158.44732297063902,169.16326530612244,177.40712945590994,159.61834862385322,152.26796407185628,159.74003795066415,176.47672253258847,198.5774193548387,163.35616438356163
LSG,null,null,null,null,null,170.83219178082192,168.8667883211679,178.11111111111111,201.32
MI,164.24416796267496,170.48446069469836,166.76565008025682,172.42459016393443,157.79949421965318,159.57169459962756,184.19639934533552,184.843984962406,155.0
PBKS,162.1640625,160.71946564885496,173.53539019963702,166.7462962962963,154.53483992467042,168.94686907020872,182.93274336283187,177.5230202578269,243.0
RCB,144.03917525773196,168.2467043314501,166.2509881422925,153.19618055555554,158.58444364422039,167.18524590163935,179.12264150943398,200.21313868613137,187.0
RPS,155.9670510708402,null,null,null,null,null,null,null,null


team_name,2017,2018,2019,2020,2021,2022,2023,2024,2025
CSK,null,211,179,200,220,216,235,212,158
DC,214,219,213,228,198,215,213,257,211
GL,208,null,null,null,null,null,null,null,null
GT,null,null,null,null,null,199,233,231,232
KKR,187,245,232,210,202,210,207,272,174
LSG,null,null,null,null,null,211,257,214,209
MI,223,213,198,208,235,190,218,247,155
PBKS,230,214,197,223,221,209,214,262,243
RCB,213,218,213,201,204,207,218,262,196
RPS,187,null,null,null,null,null,null,null,null
